# KAVI audience enrichment

Answers the finder's question *"Who is it for?"* for every resource. Run the cells from top to bottom.

The notebook reads `data/cleaned/resources_extended.csv` (from `scripts/pdf/extend_clean_resources.py`) and writes:

- `data/cleaned/resource_audiences.csv` - one row per `resource_id`, to be joined onto the resource table
- `data/review/audience_review.csv` - a random sample for a manual check of the rules
- `data/review/audience_no_match.csv` - rows where the description matched no audience, for a manual look

## Rules

1. **`pdf_audiences` stays as it is.** It comes from the PDF's audience chapters and is never changed or removed.
2. **Audiences from the description** are added in `description_audiences`, including two the PDF has no chapter for (*Survivors of violence*, *People without homes*).
3. **`Everyone`** is added to every resource that is *not restricted* to certain people. A resource can be `Everyone` and `Youth` at the same time: open to all, with something specific for youth.
4. **Ranking:** when a user picks an audience, e.g. *Youth*, resources tagged *Youth* come first, then resources tagged *Everyone*. Section 4 has the function.

In [ ]:
import os
import re

import pandas as pd

IN_FILE = "../../data/cleaned/resources_extended.csv"
OUT_FILE = "../../data/cleaned/resource_audiences.csv"
REVIEW_FILE = "../../data/review/audience_review.csv"
NO_MATCH_FILE = "../../data/review/audience_no_match.csv"
os.makedirs(os.path.dirname(REVIEW_FILE), exist_ok=True)

df = pd.read_csv(IN_FILE, dtype=str, keep_default_na=False)
pd.set_option("display.max_colwidth", 200)
print(len(df), "rows,", (df["pdf_audiences"] != "").sum(), "with pdf_audiences")

## 1. Audience patterns

One regular expression per audience, matched case-insensitively against the description. The names of the first nine are the PDF's audience chapters, so they line up with `pdf_audiences`.

Patterns find *mentions*. Section 2 decides whether a mention is a restriction.

In [ ]:
AUDIENCES = {
    "Youth": r"\byouth|young (people|adults|men|women)|\bteens?\b|teenagers?|adolescents?|\bminors?\b|ages? 1[0-9]\b",
    "Women": r"\bwomen\b|\bgirls?\b",
    "LGBTQI+": r"lgbt|transgender|\btrans\b|queer|gender[- ](non-?conforming|fluid|expansive)|two[- ]spirit",
    "Veterans": r"veterans?\b|service members",
    "Immigrants": r"immigrants?\b|undocumented|asylum|refugees?\b|migrants?\b",
    "Older People": r"\bseniors?\b|older (adults|people|new yorkers)|elderly|ages? (6[0-9]|5[5-9]) (and|or) (older|over|up)",
    "People with Disabilities": r"disabilit|\bdeaf\b|\bblind\b|hard of hearing|visually impaired",
    "Parents & Caregivers": r"\b(grand)?parents?\b|caregivers?\b|guardians?\b|families with children|children of incarcerated|parenting",
    "Formerly Incarcerated People": r"formerly incarcerated|justice[- ](involved|impacted)|re-?entry|returning (citizens|home)|(currently|recently) incarcerated|(leaving|released from|after) (prison|jail)|on (parole|probation)|(impacted|affected) by (incarceration|the (criminal )?(legal|justice) system)",
    "Survivors of violence": r"domestic violence|intimate partner|dating (abuse|violence)|sexual (assault|abuse|violence)|survivors?\b|trafficking",
    "People without homes": r"homeless|unhoused|without homes|street homeless",
}

def found_audiences(text):
    return [name for name, pattern in AUDIENCES.items() if re.search(pattern, text, re.I)]

df["description_audiences"] = df["description"].apply(lambda d: "; ".join(found_audiences(d)))

## 2. Restricted or for everyone

A mention alone ("services include … meals for homeless youth") does **not** restrict a resource. It counts as **restricted** to an audience only when the description says the service is *for* that group:

- in the **first sentence**, the audience follows a cue word such as *for*, *serves*, *supports*, *to*, *open to* or *who are*, within a few words. Example: "Provides free programs **for** LGBTQIA+ **youth** ages 13–24".
- **anywhere**, the text says *only*, *exclusively* or *must be* before the audience. Example: "open **only to** veterans".
- the text gives an **age range** ("ages 15–21"). This counts as restricted even without a named audience.

Each restriction keeps the words that triggered it in `restriction_evidence`, so it can be checked.

`audience_restriction` is then:

| Value | Meaning | Gets `Everyone`? |
|---|---|---|
| `restricted` | limited to the groups in `restricted_audiences` | no |
| `mixed` | restricted *and* says it serves anyone. Needs review | yes |
| `open` | says it serves anyone ("anyone", "all New Yorkers", "regardless of" …) | yes |
| `not_stated` | says neither | yes, assumed |

`everyone_basis` records whether `Everyone` was stated in the text (`stated`) or assumed because no restriction was found (`assumed`).

In [ ]:
CUE = r"\b(for|serv(?:es|ing)|supports?|assists?|helps?|connects|works with|to|open to|dedicated to|designed for|run by and for|who are|who have been)\b"
GAP = r"(?:\W+[\w'’-]+){0,6}?\W+"
STRICT = r"\b(only|exclusively|must be)\b"
AGE_RANGE = r"\bages? \d{1,2}\s*(?:-|–|—|to|through)\s*\d{1,2}\b|\b\d{1,2}\s*(?:-|–|—|to)\s*\d{1,2}\s*years? old\b"
OPEN = r"\b(anyone|everyone|all new yorkers|all (?:ages|people|residents|individuals)|regardless of|open to (?:all|the public)|no (?:eligibility|referral) (?:requirements?|needed))\b"

def first_sentence(text):
    return re.split(r"(?<=[.!?])\s", text.strip(), maxsplit=1)[0]

def restricted_to(text):
    """Audiences the description limits the resource to, each with the words that triggered it."""
    found = {}
    first = first_sentence(text)
    for name, pattern in AUDIENCES.items():
        m = (re.search(CUE + GAP + "(?:" + pattern + ")", first, re.I)
             or re.search(STRICT + GAP + "(?:" + pattern + ")", text, re.I)
             or re.search("(?:" + pattern + r")\W+only\b", text, re.I))
        if m:
            found[name] = m.group(0)
    return found

def classify(text):
    limits = restricted_to(text)
    age = re.search(AGE_RANGE, text, re.I)
    is_open = re.search(OPEN, text, re.I)
    evidence = list(limits.values()) + ([age.group(0)] if age else [])
    if (limits or age) and is_open:
        status = "mixed"
    elif limits or age:
        status = "restricted"
    elif is_open:
        status = "open"
    else:
        status = "not_stated"
    basis = {"restricted": "", "mixed": "stated", "open": "stated", "not_stated": "assumed"}[status]
    return pd.Series({
        "restricted_audiences": "; ".join(limits),
        "audience_restriction": status,
        "restriction_evidence": " | ".join(evidence + ([is_open.group(0)] if is_open else [])),
        "everyone_basis": basis,
    })

df = df.join(df["description"].apply(classify))
df["audience_restriction"].value_counts()

## 3. Combine: PDF audiences, description audiences and `Everyone`

`audiences_all` is what the finder filters on: `pdf_audiences` first (unchanged), then the description's audiences, then `Everyone` for every resource that isn't restricted.

In [ ]:
def split(value):
    return [v.strip() for v in value.split(";") if v.strip()]

def combine(row):
    tags = split(row["pdf_audiences"]) + split(row["description_audiences"])
    if row["everyone_basis"]:
        tags.append("Everyone")
    return "; ".join(dict.fromkeys(tags))

def source(row):
    has_pdf, has_desc = bool(row["pdf_audiences"]), bool(row["description_audiences"])
    return {(True, True): "pdf+description", (True, False): "pdf", (False, True): "description"}.get((has_pdf, has_desc), "")

df["audiences_all"] = df.apply(combine, axis=1)
df["audience_source"] = df.apply(source, axis=1)
df["audience_needs_review"] = (df["audience_restriction"] == "mixed") | (df["audience_source"] == "description")

print("Everyone:", df["audiences_all"].str.contains("Everyone").sum(), "of", len(df))
print("Everyone plus a specific audience:", (df["audiences_all"].str.contains("Everyone") & (df["audience_source"] != "")).sum())
df["everyone_basis"].replace("", "(restricted)").value_counts()

## 4. Ranking for the finder

When a user picks an audience, results come in this order:

1. **Restricted to that audience**: made for exactly this group (e.g. a youth-only program)
2. **Tagged with that audience**: from the PDF or the description, but open to others too
3. **Everyone**: not restricted, no specific mention

Resources restricted to *other* groups are left out (a women-only shelter does not show up for *Veterans*). Picking `Everyone`, or nothing, shows every resource that isn't restricted.

The same logic has to go into the finder's filter (`finder/filter.js` in #16). This function is the reference for it.

In [ ]:
def rank_for(data, audience):
    """Resources for one audience, best matches first. Adds a `rank` column (1, 2 or 3)."""
    def rank(row):
        tags = split(row["audiences_all"])
        if audience in split(row["restricted_audiences"]):
            return 1
        if audience in tags:
            return 2
        if "Everyone" in tags:
            return 3
        return None
    ranked = data.assign(rank=data.apply(rank, axis=1)).dropna(subset=["rank"])
    return ranked.astype({"rank": int}).sort_values(["rank", "org_name"])

youth = rank_for(df, "Youth")
print(youth["rank"].value_counts().sort_index().rename({1: "1 restricted to Youth", 2: "2 tagged Youth", 3: "3 Everyone"}))
youth[["rank", "org_name", "audiences_all", "restriction_evidence"]].head(15)

## 5. Cross-check against the descriptions

Where the description and the PDF disagree is the quickest way to find weak patterns.

In [ ]:
pdf_only = df[(df["pdf_audiences"] != "") & (df["description_audiences"] == "")]
desc_only = df[(df["pdf_audiences"] == "") & (df["description_audiences"] != "")]
print(len(pdf_only), "rows: PDF names an audience, description does not")
print(len(desc_only), "rows: description names an audience, PDF does not")

restricted = df[df["audience_restriction"] == "restricted"]
print(len(restricted), "restricted rows. Sample, with the words that triggered it:")
restricted[["org_name", "restricted_audiences", "restriction_evidence", "description"]].sample(10, random_state=1)

## 6. Rows with no audience match

Rows where the description matched **no audience pattern**, written to `audience_no_match.csv` for a manual look:

- `no audience anywhere`: neither the PDF nor the description names one. These get `Everyone` unless an age range restricts them.
- `PDF only`: the PDF puts the organization in an audience chapter, but the description doesn't mention it. Often a gap in the patterns above.

Cross-reference rows (`row_type = cross_reference`) only say "See the full listing …", so they never match; they are listed separately at the end.

In [ ]:
no_match = df[df["description_audiences"] == ""].copy()
no_match["match_gap"] = no_match["pdf_audiences"].map(lambda p: "PDF only" if p else "no audience anywhere")
no_match["is_cross_reference"] = no_match["row_type"] == "cross_reference"
no_match = no_match.sort_values(["is_cross_reference", "match_gap", "org_name"])

print(no_match.groupby(["is_cross_reference", "match_gap"]).size())
no_match[["resource_id", "org_name", "row_type", "match_gap", "pdf_audiences", "audience_restriction", "description"]].to_csv(NO_MATCH_FILE, index=False)
print(len(no_match), "rows written to", NO_MATCH_FILE)

no_match[~no_match["is_cross_reference"]][["org_name", "match_gap", "pdf_audiences", "description"]].head(20)

## 7. Save, plus a review sample

`audience_review.csv` is a random sample of 50 rows with empty columns for a person's verdict. Check it, then sharpen the patterns above and run again.

In [ ]:
cols = ["resource_id", "org_name", "pdf_audiences", "description_audiences", "restricted_audiences",
        "audience_restriction", "restriction_evidence", "everyone_basis", "audiences_all",
        "audience_source", "audience_needs_review"]
df[cols].to_csv(OUT_FILE, index=False)

sample = df[df["row_type"] != "cross_reference"].sample(50, random_state=42)
review = sample[["resource_id", "org_name", "description", "pdf_audiences", "description_audiences",
                 "restricted_audiences", "audience_restriction", "restriction_evidence", "audiences_all"]].copy()
review["human_verdict"] = ""        # agree / change / unsure
review["human_audiences"] = ""      # only for change: the correct list, separated by "; "
review["human_restriction"] = ""    # only for change: restricted / open / not_stated
review["human_notes"] = ""
review.sort_values("resource_id", key=lambda s: s.astype(int)).to_csv(REVIEW_FILE, index=False)

print(len(df), "rows written to", OUT_FILE)
print(len(review), "rows written to", REVIEW_FILE)